##Ingest Races.csv File

######1.Read Data from races.csv using pyspark dataframe reader
######2.Add Ingestion Metdata
    a.Add ingestion timestamp
    b.Add source
######3.Write final dataframe to bronze schema

## Step1 .Read data 

In [0]:
%run ../00.common/01.environment_config

In [0]:
%run ../00.common/02.bronze_helpers

In [0]:
source_file = f"{landing_folder_path}/races.csv"
table_name = f"{catalog_name}.{bronze_schema}.races"

In [0]:
from pyspark.sql.types import StructType,StructField,StringType,FloatType,IntegerType,DateType

races_schema = StructType(
    [
        StructField('season', StringType(), True),
        StructField('round', IntegerType(), True),
        StructField('url', StringType(), True),
        StructField('raceName', StringType(), True),
        StructField('date',DateType(),True),
        StructField('circuitid', StringType(), True)
        
    ]
)

In [0]:
races_df = (
    spark.read
    .format('csv')
    .option('header',True)
    .schema(races_schema)
    .option('mode','FAILFAST')
    .load(source_file)
)

## Step2 . Add Ingestion Metadata

In [0]:
from pyspark.sql import functions as F
races_final_df = add_ingestion_metadata(races_df)

In [0]:
display(races_final_df)

## Write Data to Delta Table

##Step3 .Write Data to Delta Table

In [0]:
(
    races_final_df
    .write
    .format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
%sql
SELECT * FROM formula1.bronze.races;

In [0]:
# display(spark.read.table('formula1.bronze.circuits'))